# 📸🗣️ Automated shareable notes from videos

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/examples/conference_slide_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

In this tutorial, we'll explore an advanced yet accessible technique for retrieving visual information from video content based on spoken word queries. Specifically, we'll focus on finding information on slides in a video recording of a speech.

As video content continues to grow in volume and importance, being able to quickly find specific information within videos becomes crucial. Imagine being able to locate a particular statistic mentioned in a hour-long presentation without watching the entire video. That's the power of multimodal video search!

This approach combines VideoDB's understanding, indexing, and query APIs to create a robust multimodal retrieval pipeline. Don't worry if these terms sound complex - we'll break everything down step by step!

## Setup
---

### 📦  Installing packages

In [ ]:
!pip install -q videodb python-dotenv


### 🔑 API keys
Before proceeding, ensure access to [VideoDB](https://videodb.io). If not, sign up for API access on the respective platforms.

Enter your VideoDB API key when prompted. You can get one from the [VideoDB Console](https://console.videodb.io). Get $20 free credits. **No credit card needed**.


In [3]:
import videodb
import os
from getpass import getpass

# Prompt user for API key securely
api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

Please enter your VideoDB API Key: ··········


## Tutorial Walkthrough

---

### 📋 Step 1: Connect to VideoDB

Gear up by establishing a connection to VideoDB

In [4]:
from videodb import connect

# Connect to VideoDB using your API key
conn = connect()
coll = conn.get_collection()
print("Connected to VideoDB successfully.")

Connected to VideoDB successfully!


### 🎬 Step 2: Upload the Video

In [5]:
video = coll.upload(url="https://www.youtube.com/watch?v=IEe-5VOv0Js")

### 📸🗣️ Step 3: Understand the Video Across Modalities

Now we will use VideoDB's understanding pipeline to extract two reusable artifacts from the video:

1. `spoken_words` extracts a timestamped transcript of what the speaker says.
2. `vlm` extracts slide-aware visual descriptions from the video frames.

For exact spoken-phrase lookup, we will convert the transcript artifact into smaller timestamped transcript windows, index those windows, and match the retrieved timestamps against the slide artifact.

#### 🗣️ Step 3A: Understand Spoken Content

First, we transcribe the spoken content using the `spoken_words` analyzer.

This produces a timestamped transcript artifact that we will later convert into searchable phrase windows.

In [6]:
transcript_understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
            "config": {
                "language": "en",
            },
        }
    ],
)

transcript_understanding.wait_until_complete(timeout=3600, poll_interval=15)

transcript_output = transcript_understanding.get_analyzer("transcript").get_output()

print("Transcript status:", transcript_output.get("status"))
print("Transcript scenes:", len(transcript_output.get("scenes", [])))

Transcript status: done
Transcript scenes: 3


#### Prepare Transcript Windows

The spoken-word analyzer gives us word-level timestamps. For this notebook, we need precise phrase-level timestamps so we can match spoken moments to the slide visible at that time.

Instead of indexing the raw transcript artifact directly, we create smaller timestamped transcript windows from the word-level transcript. This keeps the retrieval timestamps tight enough to align with the slide records.

In [7]:
import re


def normalize_text(text):
    return re.sub(r"[^\w\s]", "", text).lower().strip()


def flatten_transcript_entries(transcript_output):
    transcript = []

    for scene in transcript_output.get("scenes", []):
        data = scene.get("data", {})

        entries = data.get("entries") or data.get("words") or []
        if entries:
            transcript.extend(entries)
            continue

        text = data.get("text")
        if text:
            transcript.append(
                {
                    "start": scene.get("start"),
                    "end": scene.get("end"),
                    "text": text,
                }
            )

    seen = set()
    deduped = []

    for item in transcript:
        key = (item.get("start"), item.get("end"), item.get("text"))
        if key in seen:
            continue
        seen.add(key)
        deduped.append(item)

    return deduped


def build_transcript_windows(transcript, window_size=32, step_size=8):
    transcript_words = [
        item
        for item in transcript
        if item.get("start") is not None
        and item.get("end") is not None
        and item.get("text")
    ]

    windows = []

    for i in range(0, len(transcript_words), step_size):
        window = transcript_words[i : i + window_size]

        if not window:
            continue

        text = " ".join(item.get("text", "") for item in window).strip()

        if not text:
            continue

        windows.append(
            {
                "start": float(window[0]["start"]),
                "end": float(window[-1]["end"]),
                "text": text,
                "normalized_text": normalize_text(text),
            }
        )

    return windows


transcript = flatten_transcript_entries(transcript_output)
transcript_windows = build_transcript_windows(transcript)

print("Transcript words:", len(transcript))
print("Transcript windows:", len(transcript_windows))
print(transcript_windows[:3])

Transcript words: 4826
Transcript windows: 604
[{'start': 2.24, 'end': 14.6, 'text': "Thank you. And thank you for joining us at Sessions today. Hey, I'm going to talk about how Stripe builds APIs and teams, so. 2019 marks the 50th anniversary of the Apollo", 'normalized_text': 'thank you and thank you for joining us at sessions today hey im going to talk about how stripe builds apis and teams so 2019 marks the 50th anniversary of the apollo'}, {'start': 4.08, 'end': 18.36, 'text': "at Sessions today. Hey, I'm going to talk about how Stripe builds APIs and teams, so. 2019 marks the 50th anniversary of the Apollo Moon Land. And when NASA wrote the original", 'normalized_text': 'at sessions today hey im going to talk about how stripe builds apis and teams so 2019 marks the 50th anniversary of the apollo moon land and when nasa wrote the original'}, {'start': 6.0, 'end': 21.48, 'text': "about how Stripe builds APIs and teams, so. 2019 marks the 50th anniversary of the Apollo Moon Land. An

#### 👁️ Step 3B: Understand Slide Content

Next, we analyze the visible slide content with the VLM analyzer.

We ask the model to extract readable slide text, diagrams, bullet points, titles, and key visual concepts. The output uses a structured `slide_content` field so the query pipeline can display what was visible on screen during matching spoken moments.

In [8]:
slides_understanding = video.understand(
    segmentation={
        "type": "time",
        "seconds": 30,
    },
    analyzers=[
        {
            "type": "vlm",
            "name": "slides",
            "config": {
                "model": "pro",
                "prompt": (
                    "Look at this video segment and extract the visible slide or screen content. "
                    "Focus on readable text, titles, bullet points, diagrams, and key visual concepts. "
                    "If multiple slides appear in this segment, summarize only the slide content that is most prominent. "
                    "If no meaningful slide or screen content is visible, write None."
                ),
                "schema": {
                    "slide_content": "string",
                },
            },
        }
    ],
)

slides_understanding.wait_until_complete(timeout=3600, poll_interval=15)

slides_output = slides_understanding.get_analyzer("slides").get_output()

print("Slides status:", slides_output.get("status"))
print("Slide scenes:", len(slides_output.get("scenes", [])))

Slides status: done
Slide scenes: 61


#### Preview Slide Content

Let's preview the extracted slide content before building the query pipeline.

In [9]:
def preview_slide_content(slides_output, limit=10):
    for i, scene in enumerate(slides_output.get("scenes", [])[:limit], 1):
        data = scene.get("data", {})
        print(f"Scene {i}: {scene.get('start')} - {scene.get('end')}")
        print(data.get("slide_content", ""))
        print("----")


preview_slide_content(slides_output)

Scene 1: 0.0 - 30.0
stripe SESSIONS (branding)
How Stripe builds APIs and teams
David Singleton, CTO
----
Scene 2: 30.0 - 60.0
Title: Display (DSKY)
Subtitle: Apollo Guidance Computer
Visual: Photo of the Apollo DSKY unit (keypad and small display) shown on the left of the slide.
----
Scene 3: 60.0 - 90.0
Large black-and-white photograph filling the slide: a historic mission-control/command-center scene with a large group of engineers/technicians (mostly men in shirts and ties) gathered around rows of consoles and equipment. No prominent readable slide text visible in the main image.
----
Scene 4: 90.0 - 120.0
Top of slide shows a large black-and-white historical photograph of a mission control room (rows of consoles and operators). The slide background is a pale gradient with a large angled grey trapezoid/rectangular shape on the left. No visible title, bullet text, or other readable captions are present.
----
Scene 5: 120.0 - 150.0
Title: "Core rope memory"
Body text: "1500 bits per 

#### Build Slide Records

Now we convert the VLM output into simple timestamped slide records.

Each record includes:

- `start`: scene start time
- `end`: scene end time
- `slide_content`: VLM-extracted slide content

These timestamped records will be matched against transcript query result timestamps.

In [10]:
def build_slide_records(slides_output):
    slide_records = []

    for scene in slides_output.get("scenes", []):
        data = scene.get("data", {})

        slide_records.append(
            {
                "start": float(scene.get("start", 0)),
                "end": float(scene.get("end", 0)),
                "slide_content": data.get("slide_content", ""),
            }
        )

    return slide_records


slide_records = build_slide_records(slides_output)

print("Slide records:", len(slide_records))
print(slide_records[:3])

Slide records: 61
[{'start': 0.0, 'end': 30.0, 'slide_content': 'stripe SESSIONS (branding)\nHow Stripe builds APIs and teams\nDavid Singleton, CTO'}, {'start': 30.0, 'end': 60.0, 'slide_content': 'Title: Display (DSKY)\nSubtitle: Apollo Guidance Computer\nVisual: Photo of the Apollo DSKY unit (keypad and small display) shown on the left of the slide.'}, {'start': 60.0, 'end': 90.0, 'slide_content': 'Large black-and-white photograph filling the slide: a historic mission-control/command-center scene with a large group of engineers/technicians (mostly men in shirts and ties) gathered around rows of consoles and equipment. No prominent readable slide text visible in the main image.'}]


#### Index the Transcript Windows

Now we index the smaller transcript windows. This gives us a searchable transcript index with tighter timestamps than indexing the raw transcript artifact directly.

For exact phrase lookup, we query the normalized transcript text with a `contains` filter.

In [11]:
from uuid import uuid4

TRANSCRIPT_INDEX_NAME = f"conference_transcript_windows_{uuid4().hex[:8]}"
INDEX_READY_STATUSES = {"ready", "done"}
INDEX_ACTIVE_STATUSES = {"building", "processing"}


def wait_until_index_ready(video, index, timeout=1800, poll_interval=10):
    import time

    deadline = time.time() + timeout

    while time.time() < deadline:
        latest_index = video.get_index(index_id=index.index_id)
        status = latest_index.status

        if status in INDEX_READY_STATUSES:
            return latest_index

        if status not in INDEX_ACTIVE_STATUSES:
            raise RuntimeError(f"Index build ended with status: {status}")

        time.sleep(poll_interval)

    raise TimeoutError(f"Index was not ready within {timeout} seconds.")



transcript_window_index = video.index(
    name=TRANSCRIPT_INDEX_NAME,
    source=transcript_windows,
    use_for=["semantic", "query"],
    fields={
        "semantic": ["text"],
        "text": ["text", "normalized_text"],
        "filter": ["normalized_text"],
    },
)

transcript_window_index = wait_until_index_ready(video, transcript_window_index)

print(
    "Transcript window index:",
    transcript_window_index.index_id,
    transcript_window_index.status,
)

Transcript window index: de2e7dfb37654321 ready


### Step 4: Query Pipeline Implementation

The heart of this notebook is a timestamp-alignment pipeline:

1. Query the transcript-window index for the user's spoken phrase.
2. Extract the matching transcript timestamps.
3. Find slide records whose timestamps overlap with those spoken moments.
4. Return the slide content and timeline ranges for playback.

This lets us answer questions like: "What was on screen when the speaker talked about a hard and fast rule?"

In [12]:
def simple_filter_scenes(time_ranges, scene_dicts):
    def overlaps(scene, range_start, range_end):
        scene_start = float(scene["start"])
        scene_end = float(scene["end"])
        return scene_start < range_end and scene_end > range_start

    filtered_scenes = []

    for start, end in time_ranges:
        filtered_scenes.extend(
            scene for scene in scene_dicts if overlaps(scene, start, end)
        )

    seen = set()
    deduped = []

    for scene in filtered_scenes:
        key = (scene["start"], scene["end"], scene.get("slide_content", ""))
        if key in seen:
            continue
        seen.add(key)
        deduped.append(scene)

    return deduped

In [13]:
def query_pipeline(query, video, slide_records, index_name=TRANSCRIPT_INDEX_NAME):
    normalized_query = normalize_text(query)

    transcript_results = video.query(
        index_name=index_name,
        filter=[
            {
                "field": "normalized_text",
                "op": "contains",
                "value": normalized_query,
            }
        ],
        limit=10,
        return_fields=["text", "normalized_text"],
    )

    transcript_shots = transcript_results.shots
    time_ranges = [(shot.start, shot.end) for shot in transcript_shots]

    matching_slides = simple_filter_scenes(time_ranges, slide_records)

    result_text = "\n\n".join(
        slide["slide_content"]
        for slide in matching_slides
        if slide.get("slide_content", "").lower().strip() not in {"", "none"}
    )

    result_timeline = [(slide["start"], slide["end"]) for slide in matching_slides]

    return result_text, result_timeline, transcript_shots

### Step 5: View the Query Results

Now let's query for a phrase from the talk. The pipeline finds transcript moments where the speaker discusses the phrase, then returns the slide content visible during those moments.

In [14]:
from videodb import play_stream

query = "hard and fast rule"

result_text, result_timeline, transcript_shots = query_pipeline(
    query,
    video,
    slide_records,
)

if result_timeline:
    stream_link = video.generate_stream(result_timeline)
    play_stream(stream_link)
else:
    print("No matching slide timeline found.")

print("Matched spoken timestamp ranges:")
print([(shot.start, shot.end) for shot in transcript_shots])

print("\nMatched transcript windows:")
for shot in transcript_shots:
    metadata = getattr(shot, "metadata", {}) or {}
    print(f"{shot.start:.2f}s - {shot.end:.2f}s: {metadata.get('text', '')}")

print("\nSlide content visible during the query:")
print(result_text)

Matched spoken timestamp ranges:
[(1203.56005859375, 1217.56005859375), (1208.5999755859375, 1219.43994140625), (1211.199951171875, 1222.52001953125), (1215.6400146484375, 1225.760009765625)]

Matched transcript windows:
1203.56s - 1217.56s: to be maintaining multiple. Paths for very long. What may seem like small detail. Are, in aggregate, highly impactful, of course. No one of these is a hard and fast rule. There
1208.60s - 1219.44s: What may seem like small detail. Are, in aggregate, highly impactful, of course. No one of these is a hard and fast rule. There can often be good reasons to make an
1211.20s - 1222.52s: aggregate, highly impactful, of course. No one of these is a hard and fast rule. There can often be good reasons to make an exception. And the API review team at Stripe
1215.64s - 1225.76s: these is a hard and fast rule. There can often be good reasons to make an exception. And the API review team at Stripe holds in person discussions. Specifically. To examine the

Slide 

The result shows the slide content visible when the speaker discusses the query. The generated stream plays the matching slide moments from the original video.

## Conclusion

This notebook demonstrates a multimodal retrieval workflow using VideoDB's understanding, indexing, and query pipeline.

We used:

- `spoken_words` to extract a timestamped transcript.
- `vlm` to extract timestamped slide content.
- `video.index()` to make custom transcript windows queryable.
- `video.query()` to retrieve spoken moments for an exact phrase.
- Timestamp overlap logic to return the slide content visible during those spoken moments.

This pattern is useful for conference talks, lectures, webinars, recorded meetings, and any video where users remember what was discussed but want to recover what was shown on screen.

## Further Resources

- [Understanding Artifacts](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/understanding-artifacts)
- [Create an Index](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/create-an-index)
- [Search and Retrieval](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/search-and-retrieval/natural-language-query)

To learn more about VideoDB's understanding and retrieval pipeline, explore:

- [Scene Index Quickstart](../quickstart/Scene%20Index%20QuickStart.ipynb)
- [Advanced Visual Search](../guides/scene-index/advanced_visual_search.ipynb)
- [Custom Annotation Pipelines](../guides/scene-index/custom_annotations.ipynb)

If you have questions or feedback, reach out through:

- [Discord](https://discord.gg/py9P639jGz)
- [GitHub](https://github.com/video-db)
- [VideoDB](https://videodb.io)
- [Email](mailto:ashu@videodb.io)
